# Kodlama Destekli Yakın Okuma — Yeniden Üretim Defteri

[![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sabricanatamanoder/KaraKitapIncelemesi/blob/main/kodlama_destekli_yakin_okuma.ipynb)

*Kara Kitap* hakkındaki Goodreads yorumlarının Jaussçu alımlama çözümlemesi.
Bu defter, makalenin **üçüncü basamağına** (kodlama destekli yakın okuma) ait
bütün sayıları ham veriden yeniden üretir.

**Nasıl çalıştırılır:** Runtime → Run all. Hücreler sırayla çalıştırılmalıdır.
Toplam süre bir dakikanın altındadır; ek kurulum gerekmez.

**Defterin ürettiği çıktılar**

| Hücre | Makaledeki karşılığı |
|---|---|
| 2 | Korpus: 872 → 770 yorum |
| 3 | Kural tabanlı kodlayıcı (altı eksen, yirmi dört etiket) |
| 4 | Erişim ve aracılık ekseninin elle doğrulanmış hâli |
| 5 | Güvenilirlik: Cohen κ, kesinlik, duyarlılık, eksen bazında duyarlılık |
| 6 | Tablo 2 — korpusun dönemsel görünümü |
| 7 | Tablo 3 — okuma deneyimi ekseni × yıldız bandı, ki-kare |
| 8 | Tablo 4 — uzak okumanın iddialarının sınanması |
| 9 | Dönemsel dönüşüm ve kanonlaşma savının sınanması |
| 10 | Yıldız ile kapanmamış mesafe |

**Not.** Kodlayıcı kural tabanlıdır: sözlükler, olumsuzlama kuralları ve eşikler
aşağıdaki hücrelerde açık hâlde durur, hiçbiri dışarıdan yüklenmez. Aynı girdi
her çalıştırmada aynı çıktıyı verir; rastgelelik yoktur.

## 1. Veri

Veri dosyası her kayıt için yorum metni, dil, yıldız puanı, tarih ve kayıt
numarası taşır. Aşağıdaki `VERI_URL` sabiti doldurulduğunda dosya doğrudan
indirilir; boş bırakılırsa Colab yükleme penceresi açılır.

In [ ]:
VERI_URL = 'https://raw.githubusercontent.com/sabricanatamanoder/KaraKitapIncelemesi/main/full_analysis_FINAL.xlsx'
DOSYA    = 'full_analysis_FINAL.xlsx'

import os, re, io, warnings
import pandas as pd, numpy as np
from scipy.stats import chi2_contingency, fisher_exact
warnings.filterwarnings('ignore')
pd.set_option('display.width', 160)

if VERI_URL:
    import urllib.request
    urllib.request.urlretrieve(VERI_URL, DOSYA)
elif not os.path.exists(DOSYA):
    try:
        from google.colab import files
        up = files.upload()
        DOSYA = list(up)[0]
    except ImportError:
        raise SystemExit('Veri dosyasını defterin yanına koyun ya da VERI_URL doldurun.')

ham = pd.read_excel(DOSYA)
print('ham kayıt         :', len(ham))
print('dil dağılımı      :', ham.language_final.value_counts().to_dict())

## 2. Kural tabanlı kodlayıcı

Şema altı eksen ve yirmi dört etiketten oluşur. Her etiket, dile göre ayrı
tanımlanmış anahtar ifade listeleriyle aranır; `NEG` sözlüğü, ifadenin geçtiği
yerde okurun onu olumsuzlayıp olumsuzlamadığını denetler.

İki dile özgü işlem burada görünür hâldedir:

* `fold()` diyakritikleri düşürür, böylece "agir" yazan yorumla "ağır" yazan
  yorum aynı kalıba takılır.
* `low()` küçük harfe çevirmeyi dile göre koşullandırır. Türkçede `I → ı`,
  İngilizcede `I → i` olmalıdır; tek kural iki dile birden uygulandığında
  `\bistanbul\b` ya da `\bidentit` gibi İngilizce kalıplar sessizce çalışmaz
  hâle gelir.

In [ ]:
FOLD = str.maketrans('çğıöşüâîûÇĞİÖŞÜÂÎÛ', 'cgiosuaiucgiosuaiu')


def fold(s):
    """Diakritikleri duser: 'agir' yazan yorumla 'ağır' yazani ayni kalip yakalasin."""
    return s.translate(FOLD)


def low(s, lang='tr'):
    """Kucultme. Turkce metinde I->ı, İ->i eslemesi yapilir; Ingilizce metinde
    duz lower() kullanilir, aksi halde 'Istanbul' ve 'I' bozulur."""
    s = str(s)
    if lang == 'tr':
        s = s.replace('I', 'ı').replace('İ', 'i')
    else:
        s = s.replace('İ', 'i')
    return fold(s.lower())

# ---------------------------------------------------------------- etiket sozlukleri
# her etiket: {'tr': [regex...], 'en': [regex...]}
LABELS = {
 # A -- okuma deneyimi
 'A1': {  # zorluk ve emek
  'tr': [r'zorlan', r'zorlaş', r'zorlayıc', r'zor bir (kitap|roman|okuma|metin)', r'okuması zor',
         r'\bağır bir\b', r'ağır ilerl', r'ağır (geldi|gelen|geliyor|ve )', r'\bağır bul',
         r'sindir(mesi|meden|erek)', r'bitirmem?.{0,18}(uzun sürdü|zor)', r'asla bitireme',
         r'zor (kitap|roman|bir eser|bir deneyim|bir dil)', r'bunalt', r'sindirmek', r'karmaşık bir dil',
         r'\bodaklanma\b', r'\bkonsantre', r'lost at times', r'\bmeşakkat', r'yordu\b', r'yoruc', r'yorucu', r'çaba (gerekt|iste)',
         r'emek (iste|gerekt)', r'sindire sindire', r'kolay (değil|bir kitap değil)', r'anlamak(ta| için) (zor|güç)',
         r'\bgüç bir\b', r'kafa yor', r'dikkat (iste|gerekt)'],
  'en': [r'\bdifficult', r'\bhard to (follow|read|get|finish)', r'\bstruggl', r'\bslog\b', r'\bdense\b',
         r'\bchalleng', r'\bdemanding\b', r'not an easy (read|book)', r'\beffort\b', r'took me (a|so|forever|ages|months|weeks)',
         r'\bconfusing\b', r'\btedious\b', r'\bpatience\b', r'\bwork to read\b', r'heavy going', r'hard going'],
 },
 'A2': {  # yarida birakma
  'tr': [r'yar[ıi][md]a? bırak', r'yarım kal', r'bitiremedim', r'bitiremiyorum', r'okuyamadım', r'tamamlayamadım',
         r'bıraktım', r'devam edemedim', r'elimden bırak(tım|mak zorunda)', r'yarım kaldı', r'rafa kaldır'],
  'en': [r'\bdnf\b', r'[iı]\s+did not finish', r"[iı]\s+didn'?t finish", r"couldn'?t finish", r'unable to finish',
         r'[iıw]\w*\s+gave up(?!\s+(trying|everything))', r'gave up (on|after|at|halfway|reading)', r'giving up (on|after)', r'abandon(ed|ing)?\s+(it|this|the book|reading|halfway)', r'had to abandon', r'put it down', r'stop(ped)? reading', r'never finished',
         r"couldn'?t get (through|past|into)", r'quit (reading|halfway|this)'],
 },
 'A3': {  # zorlugun odulu -- zorluk acikca degere cevriliyor
  'tr': [r'zor ama', r'zorlu ama', r'ama (buna|bu zahmete|değdi)', r'değer(di)?\b.{0,25}(zahmet|emek|çaba)',
         r'zahmete değ', r'emeğe değ', r'bağımlılık yap', r'zorluğu.{0,30}(güzel|keyif|zevk|değer)',
         r'(zorlan|yorul)\w*.{0,60}(ama|fakat|yine de).{0,60}(sev|beğen|değer|muhteşem|harika|bayıl)',
         r'sindire sindire.{0,40}(değer|güzel|keyif)', r'her okuyuşta', r'kaç kez okursam'],
  'en': [r'(difficult|hard|challeng|dense|demanding)\w*.{0,70}(but|yet|however).{0,70}(worth|reward|brilliant|beautiful|love|masterpiece|amazing)',
         r'worth the (effort|struggle|work|patience)', r'beautifully difficult', r'difficult but (worth|beautiful|brilliant)', r'rewarding', r'rewards? (the|patient|careful)',
         r'repays? (re-?reading|careful|the)', r'\bre-?read(s|ing)? it\b.{0,40}(reward|new|more)'],
 },
 'A4': {  # yeniden okuma
  'tr': [r'tekrar oku', r'yeniden oku', r'defalarca oku', r'tekrardan oku', r'ikinci (kez|kere|defa|okuyuş)', r'üçüncü (kez|kere|defa|okuyuş)',
         r'yıl sonra.{0,25}okud', r'tekrar okuyac', r'yeniden okuyac', r'tekrar okumak', r'yeniden okumak',
         r'her okuyuşta', r'kaç kez okursam', r'defalarca okud', r'yeniden okunmayı'],
  'en': [r're-?read', r'reread', r'second (time|reading)', r'third (time|reading)', r'read (it )?again',
         r'over and over', r'years later.{0,30}read', r'read this twice', r'read it twice', r'rereading'],
 },
 # B -- bicim ve anlati teknigi
 'B1': {  # uslup ve cumle  (siirsel dil / pratik dil karsitligi)
  'tr': [r'cümle', r'cümle yapı', r'uzun cümle', r'devrik', r'noktalama', r'virgül',
         r'anlatım bozuk', r'düşük cümle', r'üslub', r'üslup', r'dili (ağır|zor|güzel|akıcı|akıcı değil)',
         r'akıcı değil', r'akıcılık', r'türkçesi', r'kelime seçim', r'betimleme'],
  'en': [r'\bprose\b', r'sentence(s| structure| length)', r'\bstyle\b', r'\bsyntax\b', r'run-?on',
         r'punctuation', r'\bcomma', r'\blyrical\b', r'\bwriting is\b', r'\bwrites beautifully\b',
         r'\bdescriptions?\b', r'\blanguage is\b', r'\bwordy\b', r'\bverbose\b'],
 },
 'B2': {  # dagiiniklik / butunluk / olay orgusu
  'tr': [r'dağınık', r'bütünlük', r'kopuk', r'savrul', r'konudan konuya', r'sapma', r'odağını yitir',
         r'olay örgüsü', r'kurgu(su|yu|sunu)? (dağ|zayıf|karmaşık|sağlam)', r'bir sonuca bağlan',
         r'ne anlatmak isted', r'saçılmış', r'toparla(n|ya)m', r'alakasız', r'başıboş', r'gereksiz uzun',
         r'fazla uzun', r'uzun tutulmuş', r'kendi(si)?ni tekrar', r'ne dediği anlaşılm'],
  'en': [r'\btangent', r'\bdigress', r'\brambl', r'\bmeander', r'\bplot\b', r'\bdisjointed\b',
         r'\bincoherent\b', r'\bstructure\b', r'\bgoes nowhere\b', r'\bunfocused\b', r'\brepetitive\b',
         r'\brepeats\b', r'\btoo long\b', r'longer than it (needed|had to)', r'\bbloated\b', r'\bself-?indulgent\b', r'lost the (thread|plot)',
         r'unanswered questions', r'\bloose ends\b', r'\bpruned\b', r'\bvignettes\b', r'\bvery slow\b', r'\bslow\b(?=[^.]{0,30}(pace|going|burn|read))'],
 },
 'B3': {  # yapi ovgusu
  'tr': [r'kurgusu (mükemmel|muhteşem|harika|sağlam|müthiş|kusursuz)', r'yapısı(nı|na)? (hayran|muhteşem|mükemmel)',
         r'iç içe geç', r'katman', r'metinlerarası', r'üstkurmaca', r'ustalık', r'ustaca',
         r'(anlatı|kurgu|yapı)\w*.{0,25}(başyapıt|şaheser|ustalık)'],
  'en': [r'\bmasterful', r'\bintricate\b', r'\blayered\b', r'\blayers\b', r'\bmetafict',
         r'\bstructur\w+.{0,25}(brilliant|genius|masterful|remarkable)', r'\bnested\b',
         r'story within a story', r'\blabyrinth', r'\bvirtuos'],
 },
 # C -- tema ve izlek
 'C1': {  # kimlik / kendi olamamak
  'tr': [r'kimlik', r'kendi(si)? ol(a|ma)', r'başkası ol', r'benlik', r'kim olduğ', r'öteki ol',
         r'kendini ara', r'kendi olma', r'varoluş'],
  'en': [r'\bidentit', r'\b(be|being|been)\s+(one|him|her|our|your|them|my)sel(f|ves)\b', r'true to yourself', r'becoming someone else', r'\bself-?hood\b',
         r'who (he|she|we|you) (really )?(is|are)\b', r'\bthe self\b', r'\bexistential'],
 },
 'C2': {  # dogu-bati / taklit
  'tr': [r'doğu.{0,15}batı', r'batı.{0,15}doğu', r'batılılaş', r'taklit', r'öykün', r'modernleş',
         r'doğulu', r'batılı olma'],
  'en': [r'east.{0,15}west', r'west.{0,15}east', r'\boriental', r'\bimitat', r'\bwesterniz',
         r'\bmodernit', r'\bmodernis[a-z]*tion\b'],
 },
 'C3': {  # istanbul / mekan / bellek
  'tr': [r'istanbul', r'\bşehir', r'\bşehr[iîae]', r'sokak', r'boğaz', r'nişantaşı', r'beyoğlu', r'mekân', r'mekan',
         r'hafıza', r'bellek', r'nostalj', r'hüzün', r'hüzzam'],
  'en': [r'\bistanbul\b', r'\bthe city\b', r'\bstreets?\b', r'\bbosphorus\b', r'\bhuzun\b', r'\bmelanchol',
         r'\bmemory\b', r'\bnostalg'],
 },
 'C4': {  # tasavvuf / hurufilik
  'tr': [r'tasavvuf', r'hurufi', r'mevlan[aâ]', r'mesnevi', r'şeyh galip', r'hüsn.{0,3}ü aşk',
         r'sufi', r'rumi', r'attar', r'mantık.?ut.?tayr', r'simurg', r'derviş', r'ebced', r'harf(ler)?in gizem'],
  'en': [r'\bsufi', r'\bhurufi', r'\brumi\b', r'\bmesnevi\b', r'\bmasnavi\b', r'\bdervish', r'\bmystic',
         r'\bsheikh galip\b', r'\battar\b', r'conference of the birds', r'\bsimurgh\b'],
 },
 'C5': {  # yazi / ustkurmaca / okuma uzerine
  'tr': [r'yazı yaz', r'yazar olma', r'köşe yaz', r'yazma eylem', r'okuma eylem', r'anlatı üzerine',
         r'hikâye anlat', r'hikaye anlat', r'metin üzerine', r'kitap üzerine kitap'],
  'en': [r'\bwriting (itself|about writing)\b', r'\bcolumn(s|ist)\b', r'about (writing|reading|storytelling)',
         r'\bstory ?telling\b', r'\bnarrative itself\b', r'book about books'],
 },
 # D -- erisim ve aracilik
 'D1': {  # ceviri sorunu
  'tr': [r'çeviri', r'çevirmen', r'tercüme'],
  'en': [r'\btranslat'],
 },
 'D2': {  # kulturel bilgi eksikligi
  'tr': [r'(bilgi|birikim).{0,30}(olmadan|olmazsa|gerek|lazım|şart)', r'osmanlı tarihi.{0,20}bil',
         r'türk (tarihi|kültürü).{0,20}bil', r'kültürel (bilgi|arka)'],
  'en': [r'(knowledge|familiar\w*|background|understanding).{0,40}(turkish|turkey|ottoman|islam|sufi|history|culture)',
         r'(turkish|ottoman|islamic).{0,30}(history|culture|context).{0,40}(need|help|require|lack)',
         r'\bdearth of knowledge\b', r'know\w*\s+(too\s+)?little about', r'written for the turkish', r'not (being )?turkish', r'if (i|you) (were|knew) more (versed|familiar)',
         r'more versed in'],
 },
 'D3': {  # anadilde okuma ayricaligi / kaynak dil vurgusu
  'tr': [r'iyi ki türkçe bil', r'ana ?dil', r'kendi dilim(de|izde)?', r'kendi dilinde oku', r'türkçe (okum|okud|yazıl).{0,40}(şans|ayrıcalık|iyi ki|haz|tat)',
         r'başka dilde.{0,30}(vermez|olmaz|aynı değil)', r'orijinal(ini)? dilinde',
         r'türkçe (bilmek|okuyabilmek).{0,25}(şans|ayrıcalık|nimet)', r'çevirisinden okuyanlar'],
  'en': [r'(read|reading) (it )?in (the )?(original|turkish)', r'wish i (could read|knew) turkish',
         r'in its original language', r'lost in translation', r'must be better in turkish',
         r'(original|source) language'],
 },
 'D4': {  # yardimci kaynak / kilavuz
  'tr': [r'kara kitap.?[’\']?ın sırlar', r'nüket esen', r'üzerine yazılar', r'kullanma kılavuzu', r'okuma rehberi', r'rehber kitap',
         r'ikinci bir kitap', r'yardımcı kaynak', r'inceleme kitab', r'şerh (kitab|edil|li)',
         r'(anlamak|çözmek) için.{0,30}(başka|ikinci|yardımcı) (bir )?(kitap|kaynak)'],
  'en': [r'\bcompanion (book|guide|volume)\b', r'\bsecondary (reading|source)', r'\bannotat',
         r'\bguide to\b', r'\bfootnote', r'\bglossar', r'had to (look|google|wiki)', r'\bwikipedia\b', r'\bwiki\b'],
 },
 # E -- metin disi otorite
 'E1': {'tr': [r'nobel'], 'en': [r'\bnobel\b']},
 'E2': {  # basyapit iddiasi
  'tr': [r'başyapıt', r'şaheser', r'başeser', r'klasik(leş|tir)', r'edebiyat tarihine', r'ölmeden önce oku'],
  'en': [r'\bmasterpiece\b', r'\bmagnum opus\b', r'\bmasterwork\b', r'\bmodern classic\b', r'\ba classic\b',
         r'\bgreat(est)? nove'],
 },
 'E3': {  # pamuk kulliyati
  'tr': [r'benim adım kırmızı', r'beyaz kale', r'yeni hayat', r'masumiyet müzesi', r'sessiz ev',
         r'cevdet bey', r"\bkar['’](ı|i|ın|in|da|dan|a)\b", r'kafamda bir tuhaflık', r'kırmızı saçlı', r'veba geceleri',
         r'istanbul.{0,3}(hatıralar|şehir)', r'diğer (kitapları|romanları)', r'pamuk.{0,20}(külliyat|diğer)'],
  'en': [r'my name is red', r'the white castle', r'the new life', r'museum of innocence', r'silent house',
         r"[“\"']snow[”\"']", r'\bsnow\b(?=[^.]{0,60}(pamuk|red|castle|novel))', r'(loved|read|enjoyed|recommend|liked)\s+\W?snow\b', r'\bsnow\b(?=\s+(over|and|,))', r'strangeness in my mind', r'red-?haired woman', r'nights of plague',
         r"(pamuk'?s|his)\s+other\W{0,20}(books|novels|works)", r'other\s+(books|novels|works)\s+(by|of)\s+\w*\s*pamuk'],
 },
 'E4': {  # dunya edebiyati referansi
  'tr': [r'borges', r'calvino', r'kafka', r'proust', r'dostoyevski', r'joyce', r'eco\b', r'marquez',
         r'márquez', r'nabokov', r'dante', r'boccaccio', r'\bpoe\b', r'cervantes', r'faulkner'],
  'en': [r'\bborges\b', r'\bcalvino\b', r'\bkafka\b', r'\bproust\b', r'\bdostoev', r'\bjoyce\b',
         r'\beco\b', r'\bmarquez\b', r'\bmárquez\b', r'\bnabokov\b', r'\bdante\b', r'\bboccaccio\b',
         r'\bcervantes\b', r'\bfaulkner\b', r'\bthousand and one nights\b', r'\barabian nights\b', r'\b1001\s+(nights|stories|tales)\b', r'\bauster\b', r'new york trilogy', r'\bul+ys+es\b'],
 },
 'E5': {  # turk edebiyati referansi
  'tr': [r'tanpınar', r'oğuz atay', r'tutunamayanlar', r'yusuf atılgan', r'nahid sırrı', r'ahmet hamdi',
         r'saatleri ayarlama', r'türk edebiyatı', r'türk roman', r'bilge karasu', r'sait faik', r'yaşar kemal'],
  'en': [r'\btanpinar\b', r'\boguz atay\b', r'\bturkish (literature|novel|writer|author)\b',
         r'\byasar kemal\b', r'\byahya kemal\b'],
 },
 'E6': {  # okurun kendini yetersiz gormesi
  'tr': [r'benim (eksikliğim|kabahatim|suçum)', r'(birikimim|donanımım).{0,20}yeter(siz|medi)',
         r'ben mi anlamadım', r'anlayamadım galiba', r'belki de ben(im)? (anlamadım|anlayamadım|yetersiz|hazır değil|kabahat)', r'belki ben\b', r'\bmalim\b', r'kof bir okur', r'toyluğum', r'bana mı öyle geldi',
         r'seviyem', r'hazır değilmiş', r'erken okumuş'],
  'en': [r'\bmaybe i(\'m| am)? (just )?(not|too)', r'\bover my head\b', r'\bnot smart enough\b',
         r"\bit'?s (just )?me\b", r'\bi lack(ed)? the\b', r'\bnot well.?read enough\b',
         r'\bmy (own )?(fault|shortcoming|limitation)'],
 },
 # F -- tur beklentisi
 'F1': {  # polisiye beklentisi
  'tr': [r'polisiye', r'dedektif', r'gizem roman', r'cinayet roman', r'arka kapak.{0,40}(san|bekle)',
         r'(bekle|san)\w*.{0,30}polisiye'],
  'en': [r'\bdetective\b', r'\bmystery novel\b', r'\bcrime novel\b', r'\bnoir\b', r'\bwhodunit\b',
         r'\bthriller', r'\bwhodun', r'expect\w*.{0,40}(mystery|detective|crime)', r'(mystery|detective).{0,40}expect'],
 },
 'F2': {  # postmodern etiketleme
  'tr': [r'postmodern', r'post[- ]?modern'],
  'en': [r'\bpost-?modern'],
 },
}

# olumsuzlama pencereleri (yalnizca A1/A2/A4 icin uygulanir)
NEG = {
 'tr': [r'zorlanmadım', r'zorlanmıyor', r'zor değil', r'hiç zorlan', r'bırakmadım', r'yarıda bırakmadım',
        r'tekrar okumam', r'bir daha okumam', r'yeniden okumam', r'tekrar okumayı düşünmüyor'],
 'en': [r'not (at all )?difficult', r"wasn'?t (that )?(hard|difficult)", r'never (put it down|gave up)',
        r"didn'?t struggle", r'no trouble', r"won'?t (be )?re-?read", r'never re-?read'],
}

AXES = {'A': ['A1','A2','A3','A4'], 'B': ['B1','B2','B3'], 'C': ['C1','C2','C3','C4','C5'],
        'D': ['D1','D2','D3','D4'], 'E': ['E1','E2','E3','E4','E5','E6'], 'F': ['F1','F2']}


def is_quote_dump(txt):
    """Yalnizca romandan alinti / Kindle notu olan yorumlari isaretle."""
    t = str(txt)
    if re.search(r'your highlight on page|added on \w+day|location \d+', t, re.I):
        return True
    # tirnak icindeki karakterlerin orani
    q = sum(len(m) for m in re.findall(r'[“"“][^“”"“”]{25,}[”"”]', t))
    return len(t) > 60 and q / max(len(t), 1) > 0.6


_CACHE = {}


def _pats(lang, pats):
    key = (lang, id(pats))
    if key not in _CACHE:
        _CACHE[key] = [re.compile(fold(x)) for x in pats]
    return _CACHE[key]


def code_row(txt, lang):
    t = low(txt, lang)
    negs = [n for n in NEG[lang] if re.search(fold(n), t)]
    out = set()
    for lab, pats in LABELS.items():
        for rx in _pats(lang, pats[lang]):
            if rx.search(t):
                out.add(lab)
                break
    # A3: "... ama sevemedim/degmedi" bicimindeki ters ornekleri dusur
    if 'A3' in out and re.search(fold(r'(seveme|beğeneme|hoşlanama|değmedi)'), t) \
       and not re.search(fold(r'(zahmete değ|emeğe değ|bağımlılık yap|worth the)'), t):
        out.discard('A3')
    if negs:
        for lab in ('A1', 'A2', 'A4'):
            for n in negs:
                m = re.search(fold(n), t)
                if m and lab in out:
                    # olumsuzlanan ifade etiketin tek dayanagiysa etiketi dusur
                    stripped = t[:m.start()] + ' ' + t[m.end():]
                    if not any(re.search(p, stripped) for p in LABELS[lab][lang]):
                        out.discard(lab)
    return sorted(out)

## 3. Korpusun kurulması ve kodlanması

Alıntı yığınları ve kırk karakterin altındaki yorumlar dışarıda bırakılır:
okurun metinle ilişkisine dair kodlanabilir bir ifade taşımazlar. Puan bantları
üçe indirilir (1–2, 3, 4–5); dönemler iki korpusa da aynı takvim sınırlarıyla
uygulanır.

In [ ]:
df = ham.copy()
df['year']  = pd.to_datetime(df.date_parsed).dt.year
df['chars'] = df.comment.astype(str).str.len()
df['words'] = df.comment.astype(str).str.split().str.len()
df['quote_dump']  = df.comment.apply(is_quote_dump)
df['substantive'] = (~df.quote_dump) & (df.chars >= 40)
df['period'] = df.year.map(lambda y: 'D1' if y <= 2012 else ('D2' if y <= 2019 else 'D3'))
df['band']   = df.rating_normalized.map(
    lambda r: 'yok' if pd.isna(r) else ('1-2' if r <= 2 else ('3' if r == 3 else '4-5')))

codes = [code_row(r.comment, r.language_final) for r in df.itertuples()]
df['labels'] = ['|'.join(c) for c in codes]
for lab in LABELS:
    df[lab] = [1 if lab in c else 0 for c in codes]
for ax, labs in AXES.items():
    df['AX_' + ax] = df[labs].max(axis=1)

S = df[df.substantive].copy()
print('toplam yorum            :', len(df))
print('çözümleme dışı           :', int((~df.substantive).sum()))
print('çözümlemeye giren        :', len(S), dict(S.language_final.value_counts()))
print('puanı olmayan (bant dışı):', dict(S[S.band == 'yok'].language_final.value_counts()))

## 4. Erişim ve aracılık ekseninin elle doğrulanması

Otomatik sözlük bu eksende güvenilir değildir. Türkçede "çevir-" kökü romanın
olay örgüsündeki bir unsuru (Rüya'nın polisiye roman çevirmenliği) ve deyimleri
("sayfaları çevirmek", "eli boş çevirmez") yakalar, buna karşılık
"çevirenler / çevirdiler" biçimlerini kaçırır; İngilizcede "had to look" kalıbı
romanın içeriğine takılır.

İki korpustaki bütün adaylar tam metin okunarak sınıflandırılmıştır. Aşağıdaki
kümeler o elle kodlamanın kaydıdır; kayıt numaraları verildiği için her karar
ham veriye geri izlenebilir.

In [ ]:
TR = {
    'D1': {91, 339, 341},                       # ceviri metnin kendisi hakkinda yargi nesnesi
    'D2': {157, 307, 544},                      # kulturel bilgi eksikligi
    'D3a': {28, 341, 359, 364, 451},            # kaynak dile sahiplik
    'D3b': set(),                               # kaynak dilden yoksunluk
    # D4: okurun romani anlamak icin roman disi bir metne basvurmasi, basvurmayi
    # onermesi ya da boyle bir metnin olmasini istemesi. Iki dilde ayni olcut.
    'D4': {21, 24, 25, 45, 46, 55, 65, 72, 78, 79, 109, 157, 205, 296, 305, 335,
           337, 375, 398, 421, 454, 859},
}
EN = {
    'D2_out': {855, 545, 360},                  # yanlis pozitif: icerik / gelecege donuk istek / kulturel yakinlik
    # D4: ayni olcut. Cogunlugu cevirmen sonsozu; kalani Wikipedia ve olmayan aparat istegi.
    'D4': {6, 9, 22, 54, 58, 66, 71, 86, 185, 197, 207, 220, 226, 234, 282, 301,
           366, 384, 386, 562, 563, 579, 583, 594, 622, 871},
    'D4_afterword': {9, 22, 54, 58, 66, 86, 185, 207, 220, 226, 234, 282, 384,
                     386, 562, 563, 579, 583, 594, 622, 871},
    'D3a': {272, 570},
    'D3b': {58, 92, 94, 107, 220, 238, 386, 511, 579, 605, 638, 682, 763, 861},
}

tr = S.language_final == 'tr'
en = S.language_final == 'en'
rid = S.record_id

S.loc[tr, 'vD1'] = rid[tr].isin(TR['D1']).astype(int)
S.loc[tr, 'vD2'] = rid[tr].isin(TR['D2']).astype(int)
S.loc[tr, 'vD3'] = rid[tr].isin(TR['D3a'] | TR['D3b']).astype(int)
S.loc[tr, 'vD4'] = rid[tr].isin(TR['D4']).astype(int)

S.loc[en, 'vD1'] = S.loc[en, 'D1']
S.loc[en, 'vD2'] = (S.loc[en, 'D2'] & ~rid[en].isin(EN['D2_out'])).astype(int)
S.loc[en, 'vD3'] = rid[en].isin(EN['D3a'] | EN['D3b']).astype(int)
S.loc[en, 'vD4'] = rid[en].isin(EN['D4']).astype(int)

for c in ('vD1', 'vD2', 'vD3', 'vD4'):
    S[c] = S[c].fillna(0).astype(int)
S['vAX_D'] = S[['vD1', 'vD2', 'vD3', 'vD4']].max(axis=1)
S['D3a'] = rid.isin(TR['D3a'] | EN['D3a']).astype(int)
S['D3b'] = rid.isin(TR['D3b'] | EN['D3b']).astype(int)

T, E = S[S.language_final == 'tr'], S[S.language_final == 'en']
print(f'{"":32s}{"Türkçe":>16s}{"İngilizce":>16s}')
for nm, c in [('D1 çeviri farkındalığı', 'vD1'), ('D2 kültürel bilgi eksikliği', 'vD2'),
              ('D3 kaynak dille ilişki', 'vD3'), ('   D3a sahiplik', 'D3a'),
              ('   D3b yoksunluk', 'D3b'), ('D4 yardımcı kaynak', 'vD4'),
              ('D ekseni (birleşim)', 'vAX_D')]:
    a, b = int(T[c].sum()), int(E[c].sum())
    print(f'{nm:32s}{a:6d} (%{100*a/len(T):4.1f}){b:7d} (%{100*b/len(E):4.1f})')
print()
print('otomatik D ekseni  : tr %%%.1f  en %%%.1f' % (100*T.AX_D.mean(), 100*E.AX_D.mean()))
print('doğrulanmış D      : tr %%%.1f  en %%%.1f' % (100*T.vAX_D.mean(), 100*E.vAX_D.mean()))
print('D4 oranı tr/en     : %.1f kat' % (T.vD4.mean() / E.vD4.mean()))
print('çevirmen sonsözüne başvuran İngilizce yorum: %d / %d'
      % (len(EN['D4_afterword']), len(EN['D4'])))

## 5. Güvenilirlik

Sözlükler önce altmış yorumluk bir geliştirme örnekleminde denenmiş, ölçüm
bununla kesişmeyen bağımsız elli yorumluk (25 Türkçe, 25 İngilizce) bir sınama
örnekleminde yapılmıştır. Örneklemdeki her yorum tam metin okunarak elle
kodlanmış; aşağıdaki `GOLD` sözlüğü o referans kodlamadır.

Ölçüm, sözlüklere hiç bakılmamış bir örneklemde yapıldığı için aşırı uyum
sorunu doğmaz.

In [ ]:
GOLD = {
 # --- Turkce 25
 326:'AE', 508:'C', 551:'ABC', 591:'A', 337:'ABCEF', 248:'AB', 15:'B', 520:'E',
 262:'ABE', 671:'B', 12:'', 188:'ABE', 851:'C', 522:'BC', 353:'ABCE', 60:'AC',
 93:'ABE', 26:'ABC', 435:'AB', 88:'ABCE', 132:'AC', 127:'C', 409:'ABCE',
 397:'ABCE', 443:'',
 # --- Ingilizce 25
 611:'CE', 386:'ACD', 593:'BCD', 595:'ABCF', 583:'ABCD', 817:'', 94:'ACD',
 574:'BC', 33:'', 343:'BC', 150:'ABCEF', 166:'BE', 240:'BC', 655:'CD',
 247:'BC', 332:'ABC', 346:'ACE', 308:'ABCEF', 149:'CF', 440:'C', 712:'E',
 827:'C', 461:'E', 704:'', 255:'ABC',
}

AX = list('ABCDEF')
idx = S.set_index('record_id')
a = b = c = d = 0
per = {x: dict(tp=0, fp=0, fn=0) for x in AX}
for rid_, gold in GOLD.items():
    r = idx.loc[rid_]
    pred = {x for x in AX if r['AX_' + x] == 1}
    true = set(gold)
    for x in AX:
        p, t = x in pred, x in true
        if p and t:      a += 1; per[x]['tp'] += 1
        elif p:          b += 1; per[x]['fp'] += 1
        elif t:          c += 1; per[x]['fn'] += 1
        else:            d += 1

n = a + b + c + d
po = (a + d) / n
pe = (((a + b) * (a + c)) + ((c + d) * (b + d))) / n**2
print(f'karar sayısı        : {n}  (50 yorum × 6 eksen)')
print(f'uyum (po)           : {po:.3f}')
print(f'beklenen uyum (pe)  : {pe:.3f}')
print(f'Cohen kappa         : {(po-pe)/(1-pe):.3f}')
print(f'kesinlik            : {a/(a+b):.3f}')
print(f'duyarlılık          : {a/(a+c):.3f}')
print()
print('eksen bazında:')
for x in AX:
    v = per[x]
    pr = v['tp'] / (v['tp'] + v['fp']) if v['tp'] + v['fp'] else float('nan')
    rc = v['tp'] / (v['tp'] + v['fn']) if v['tp'] + v['fn'] else float('nan')
    print(f'  {x}: kesinlik={pr:.2f}  duyarlılık={rc:.2f}')
print()
print('Duyarlılığın en düşük olduğu A ekseni, bütün A oranlarının taban değer')
print('olarak okunmasını gerektirir.')

## 6. Tablo 2 — Korpusun dönemsel görünümü

Dönem sınırları veriden değil alımlama ortamındaki değişimlerden türetilmiştir:
2006 Nobel Ödülü ve Freely çevirisinin ardından gelen ilk dalga 2012'de
sönümlenir; 2020, Türkçe korpustaki en büyük yıllık artışın yaşandığı yıldır.
Türkçe D1 hücresi dört yorumdan oluştuğu için çözümleme dışındadır.

In [ ]:
sat = []
for lang, ad in (('tr', 'Türkçe'), ('en', 'İngilizce')):
    for p, ar in (('D1', '2007–2012'), ('D2', '2013–2019'), ('D3', '2020–2025')):
        g = S[(S.language_final == lang) & (S.period == p)]
        sat.append({
            'Korpus': ad, 'Dönem': f'{p} ({ar})', 'n': len(g),
            'Ort. puan': round(g.rating_normalized.mean(), 2) if len(g) > 5 else None,
            'Medyan uzunluk': int(g.chars.median()) if len(g) else None,
            'Etiket/yorum': round(g[list(LABELS)].sum(axis=1).mean(), 2) if len(g) > 5 else None,
            'Eksen/yorum': round((g[[f'AX_{x}' for x in 'ABCEF']].sum(axis=1)
                                  + g.vAX_D).mean(), 2) if len(g) > 5 else None})
print(pd.DataFrame(sat).to_string(index=False))
print()
print('İki korpusta da yorum uzunluğu ve yorum başına düşen etiket sayısı')
print('dönemler boyunca artmaktadır: okur söylemi hem uzamakta hem')
print('çok boyutlulaşmaktadır.')

## 7. Tablo 3 — Okuma deneyimi ekseni × yıldız bandı

Bölümün en beklenmedik bulgusu buradadır: **zorluk bildirimi puanı
öngörmemektedir.** Ayrışma zorluğun kendisinde değil, okurun zorlukla ne
yaptığındadır.

In [ ]:
def kikare(gruplar, col):
    """Etiket x grup capraz tablosunda ki-kare. Iki gruplu tabloda Fisher da doner."""
    tab = [[int(g[col].sum()), len(g) - int(g[col].sum())] for g in gruplar]
    chi2, p, dof, _ = chi2_contingency(tab)
    pf = fisher_exact(tab)[1] if len(gruplar) == 2 else None
    return chi2, p, dof, pf

BANT = ['1-2', '3', '4-5']
ETIKET = [('A1', 'Zorluk ve emek'), ('A2', 'Yarıda bırakma'), ('A4', 'Yeniden okuma'),
          ('B1', 'Üslup ve cümle'), ('B2', 'Dağınıklık')]

for lang, ad in (('tr', 'TÜRKÇE'), ('en', 'İNGİLİZCE')):
    G = [S[(S.language_final == lang) & (S.band == b)] for b in BANT]
    print(f'--- {ad}   ' + '   '.join(f'{b} (n={len(g)})' for b, g in zip(BANT, G)))
    for col, nm in ETIKET:
        yuzde = '  '.join(f'{100*g[col].mean():5.1f}' for g in G)
        chi2, p, dof, _ = kikare(G, col)
        yildiz = '*' if p < 0.05 else ' '
        print(f'  {nm:18s} {yuzde}   |  chi2={chi2:6.2f}  sd={dof}  p={p:.3f}{yildiz}')
    print()
print('Zorluk satırı iki dilde de anlamsızdır: negatif deneyim neredeyse')
print('evrenseldir. Ayrımı kuran satırlar dile göre değişir — İngilizcede')
print('yarıda bırakma ve yeniden okuma, Türkçede üslup ve dağınıklık.')

## 8. Tablo 4 — Uzak okumanın iddialarının sınanması

Uzak okuma bölümü kelime bulutlarından üç iddia üretmişti. Yakın okuma bunları
tek tek sınar: doğrular, düzeltir ya da bileşenlerine ayırır.

In [ ]:
T, E = S[S.language_final == 'tr'], S[S.language_final == 'en']

# (i) Kimlik: tema olarak var mı, sözcük olarak var mı?
kt = T.comment.apply(lambda x: bool(re.search('kimlik', low(x, 'tr'))))
ke = E.comment.apply(lambda x: bool(re.search(r'\bidentit', low(x, 'en'))))
print('(i) KİMLİK')
print(f'    tema (C1)      : tr %{100*T.C1.mean():.1f}   en %{100*E.C1.mean():.1f}'
      f'   -> oran {E.C1.mean()/T.C1.mean():.2f}')
print(f'    sözcüğü kullanan: tr %{100*kt.mean():.1f}   en %{100*ke.mean():.1f}'
      f'   (temayı sözcükle adlandırma oranı: tr %{100*kt.sum()/T.C1.sum():.0f}'
      f'   en %{100*ke.sum()/E.C1.sum():.0f})')
print(f'    temayı taşıyıp sözcüğü kullanmayan Türkçe yorum: {int((T.C1 & ~kt).sum())}')
print('    Kelime bulutundaki 12,5 kat, yorum düzeyinde 1,44 kata iner.')
print()
print('(ii) KAYNAK DİLLE İLİŞKİ ve ARACILIK')
print(f'    çeviri farkındalığı  : tr %{100*T.vD1.mean():.1f}   en %{100*E.vD1.mean():.1f}')
print(f'    sahiplik (D3a)       : tr %{100*T.D3a.mean():.1f}   en %{100*E.D3a.mean():.1f}')
print(f'    yoksunluk (D3b)      : tr %{100*T.D3b.mean():.1f}   en %{100*E.D3b.mean():.1f}')
print(f'    yardımcı kaynak (D4) : tr %{100*T.vD4.mean():.1f}   en %{100*E.vD4.mean():.1f}'
      f'   -> {T.vD4.mean()/E.vD4.mean():.1f} kat')
print()
print('(iii) YENİDEN OKUMA')
print(f'    A4 : tr %{100*T.A4.mean():.1f}   en %{100*E.A4.mean():.1f}')

## 9. Dönemsel dönüşüm ve kanonlaşma savı

Jauss'a göre kanonlaşma, yapıtla okurun ufku arasındaki estetik mesafeyi
daraltır. Aşağıdaki sınama bu öngörüyü iki dilde ayrı ayrı yoklar.

Sütunlardaki yüzdeler tek başına yeterli değildir: her karşılaştırma ki-kare ile
sınanmakta, beklenen frekansı beşin altına düşen hücrelerde Fisher kesin testi
raporlanmaktadır. Yıldızla işaretlenen satırlar p < 0,05 düzeyinde anlamlıdır.

In [ ]:
KARS = [('AX_E', 'Metin dışı otorite'), ('E1', '   Nobel'), ('E3', '   Pamuk külliyatı'),
        ('AX_C', 'Tema ekseni'), ('AX_B', 'Biçim ekseni'), ('B2', '   Dağınıklık'),
        ('B1', '   Üslup'), ('A1', 'Zorluk bildirimi'), ('A3', '   Zorluğun ödülü')]

for lang, pa, pb, ad in (('tr', 'D2', 'D3', 'TÜRKÇE (D2 → D3)'),
                         ('en', 'D1', 'D3', 'İNGİLİZCE (D1 → D3)')):
    g1 = S[(S.language_final == lang) & (S.period == pa)]
    g2 = S[(S.language_final == lang) & (S.period == pb)]
    print(f'--- {ad}   n={len(g1)} → n={len(g2)}')
    for col, nm in KARS:
        chi2, p, dof, pf = kikare([g1, g2], col)
        yildiz = '*' if p < 0.05 else ' '
        print(f' {yildiz}{nm:22s} %{100*g1[col].mean():5.1f} → %{100*g2[col].mean():5.1f}'
              f'   chi2={chi2:6.2f}  p={p:.3f}  Fisher={pf:.3f}')
    print()

print('Okunuşu: Türkçe tarafta anlamlı olan hareketler okurun neye yaslanarak')
print('konuştuğuna ilişkindir (Nobel, tema). Biçimsel şikâyetteki gerileme')
print('anlamsızdır; roman daha kolay bulunmamakta, hakkında farklı')
print('konuşulmaktadır. İngilizce tarafta metin dışı otorite hiç büyümez ve')
print('dağınıklık şikâyeti anlamlı biçimde artar. Kanonlaşmanın estetik mesafeyi')
print('daralttığı öngörüsü bu korpusta doğrulanamamaktadır.')

## 10. Yıldız, estetik mesafeyi ölçer mi?

Jauss estetik mesafeyi yapıtın ilk okur kitlesinin ortak tepkisinden okur;
Goodreads yıldızı ise tek bir okurun kitabı bitirdikten sonra verdiği hükümdür.
Son sınama, mesafeyi kapatmadığını söyleyip yine de en yüksek puanı veren
okurları arar.

In [ ]:
PT = fold(r'(anlama(dım|dığımız|dığım |dan)|anlayama(dım|dığım|yacağımız|yacağım|dığımız)|'
          r'anladığımı (san|söyleye)\w*m|kavraya\w*ma|çözemed|anlaşılmaz|'
          r'anlamak (mümkün değil|zor)|eksik (anla|kald)|ne anlatmak istedi)')
PE_ = (r"not sure (i|ı) (entirely |fully |completely |really )?understood|"
       r"didn'?t (fully |really |entirely |quite )?understand|"
       r"don'?t (fully |really |quite )?understand|still don'?t know what|no idea what|"
       r"couldn'?t understand|didn'?t (fully )?get it|over my head|"
       r"understood (very )?little|much of it (escaped|eluded)|(escaped|eluded) me")

toplam = 0
for lang, pat in (('tr', PT), ('en', PE_)):
    g = S[(S.language_final == lang) & (S.band == '4-5')]
    h = g[g.comment.apply(lambda x: bool(re.search(pat, low(x, lang))))]
    toplam += len(g)
    print(f'{lang}: 4–5 yıldız n={len(g)}  |  anlamadığını bildiren aday: {len(h)}')
    print(f'    kayıt no: {sorted(h.record_id.tolist())}')
print()
print(f'4–5 yıldız veren toplam okur: {toplam}')
print()
print('Yukarıdaki liste ADAY listesidir: kalıp, anlamamayı bildirmeyen')
print('kullanımlara da takılabilir. Adayların tamamı tam metin okunarak')
print('elenmiş ve makalede sekiz yorum raporlanmıştır. Kayıt numaraları')
print('burada verildiği için her aday ham veriden denetlenebilir.')

## Neyin yeniden üretilebildiği, neyin üretilemediği

Bu defter kodlamayı, güvenilirlik ölçümünü ve bütün nicel karşılaştırmaları
baştan üretir. Yeniden üretilemeyen üç şey vardır ve bunlar makalede de sınır
olarak kaydedilmiştir.

1. **Elle kodlama kararları.** Erişim ve aracılık eksenindeki kümeler ile
   güvenilirlik ölçümündeki `GOLD` sözlüğü, yorumların tam metni okunarak
   oluşturulmuştur. Defter bu kararları kayıt numarasıyla açık eder, dolayısıyla
   her biri denetlenebilir; ama bir betik tarafından yeniden üretilmezler.
2. **Duyarlılık sınırı.** Kodlayıcı, var olan eksenlerin yaklaşık sekizde birini
   kaçırır. Bütün oranlar taban değerdir.
3. **Okurun yazmadığı şey.** Kodlama okurun yazdığını ölçer, okuduğunu değil;
   bir yorumun samimiyeti, ironisi ya da niyeti bu yolla saptanamaz.